In [ ]:
import numpy as np
import polars as pl
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram, cophenet
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

# --------------------------------------------------------------------
# Long-form -> square matrix
#
# Your data: a Polars DataFrame with columns "source", "target", "VI"
# (long / edge-list form, one row per pair). This needs converting to
# a dense (n_cols x n_cols) symmetric numpy array before it can go
# into scipy's clustering functions.
#
# Handles the common real-world cases:
#   - only one direction of each pair present (source->target but not
#     target->source) -- typical if you computed VI(i,j) once per
#     unordered pair
#   - self-pairs (source == target) missing or present
#   - column universe inferred from the union of source/target
# --------------------------------------------------------------------

def long_to_square(df: pl.DataFrame,
                    source_col: str = 'source',
                    target_col: str = 'target',
                    value_col: str = 'VI') -> tuple[np.ndarray, list[str]]:
    # 1. full, sorted set of column names appearing on either side
    column_names = sorted(
        set(df[source_col].unique().to_list()) | set(df[target_col].unique().to_list())
    )
    idx = {name: i for i, name in enumerate(column_names)}
    n = len(column_names)

    # 2. start from zeros (diagonal correctly ends up 0 = no self-distance)
    D = np.zeros((n, n))

    # 3. fill in both directions from whichever rows we have
    rows = df.select(source_col, target_col, value_col).iter_rows()
    for s, t, v in rows:
        i, j = idx[s], idx[t]
        D[i, j] = v
        D[j, i] = v   # mirror -- safe even if df already had both directions

    return D, column_names


# --------------------------------------------------------------------
# nvi_matrix: your existing (n_cols x n_cols) normalized VI matrix.
#   - symmetric
#   - diagonal = 0  (a column has zero distance to itself)
#   - values in [0, 1]
# column_names: list of column names, same order as nvi_matrix rows/cols
# --------------------------------------------------------------------

def cluster_on_nvi(nvi_matrix, column_names, linkage_method='average'):
    n = nvi_matrix.shape[0]

    # 1. sanity checks / cleanup -----------------------------------------
    # enforce exact symmetry (guards against float drift) and zero diagonal
    D = (nvi_matrix + nvi_matrix.T) / 2
    np.fill_diagonal(D, 0)
    assert np.all(D >= -1e-9) and np.all(D <= 1 + 1e-9), "NVI must be in [0,1]"

    # 2. scipy wants the condensed (upper-triangle, no diagonal) form ----
    condensed = squareform(D, checks=False)

    # 3. hierarchical clustering -----------------------------------------
    #    average or complete linkage only -- NOT ward (assumes Euclidean)
    Z = linkage(condensed, method=linkage_method)

    # 4. check how well the dendrogram represents the original distances
    coph_corr, _ = cophenet(Z, condensed)
    print(f"Cophenetic correlation ({linkage_method} linkage): {coph_corr:.3f}")
    print("  (low value => dependency structure isn't very tree-like; "
          "cross-check against a graph-community result)")

    return Z


def cut_and_report(Z, column_names, distance_threshold=None, n_clusters=None):
    if distance_threshold is not None:
        labels = fcluster(Z, t=distance_threshold, criterion='distance')
    elif n_clusters is not None:
        labels = fcluster(Z, t=n_clusters, criterion='maxclust')
    else:
        raise ValueError("Provide either distance_threshold or n_clusters")

    clusters = {}
    for col, lab in zip(column_names, labels):
        clusters.setdefault(lab, []).append(col)

    for lab, cols in sorted(clusters.items()):
        print(f"Cluster {lab} ({len(cols)} cols): {cols}")

    return dict(zip(column_names, labels))


def plot_dendrogram(Z, column_names, title, out_path):
    plt.figure(figsize=(10, 5))
    dendrogram(Z, labels=column_names, leaf_rotation=90, color_threshold=0.5)
    plt.ylabel('Normalized VI distance')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close()


# ======================================================================
# Demo: synthetic long-form Polars DataFrame (source, target, VI),
# same three obvious column groups as before, converted via long_to_square
# ======================================================================
if __name__ == '__main__':
    np.random.seed(1)
    groups = {
        'econ': ['price', 'notional', 'fx_rate', 'yield_curve_pt'],
        'ops':  ['trade_ts', 'booking_system', 'trader_id'],
        'reg':  ['mifid_flag', 'reporting_regime', 'lei_code'],
    }
    all_cols = [c for g in groups.values() for c in g]
    group_of = {c: g for g, cols in groups.items() for c in cols}

    # build a long-form edge list -- ONE row per unordered pair, mirroring
    # how you'd typically have computed pairwise VI (e.g. itertools.combinations)
    records = []
    for i, ci in enumerate(all_cols):
        for cj in all_cols[i + 1:]:
            base = 0.15 if group_of[ci] == group_of[cj] else 0.85
            vi = float(np.clip(base + np.random.normal(0, 0.05), 0, 1))
            records.append({'source': ci, 'target': cj, 'VI': vi})

    vi_long = pl.DataFrame(records)
    print(f"Long-form input: {vi_long.shape[0]} pairs")
    print(vi_long.head(5))

    # --- the actual transform you asked for ---
    nvi_matrix, column_names = long_to_square(vi_long, 'source', 'target', 'VI')

    Z = cluster_on_nvi(nvi_matrix, column_names, linkage_method='average')

    print("\n--- Cut by distance threshold (0.5) ---")
    cluster_map = cut_and_report(Z, column_names, distance_threshold=0.5)

    print("\n--- Cut by fixed cluster count (3) ---")
    cut_and_report(Z, column_names, n_clusters=3)

    plot_dendrogram(Z, column_names,
                     'Column clustering on normalized VI (average linkage)',
                     '/home/claude/nvi_dendrogram.png')
    print("\nDendrogram saved.")